<a href="https://colab.research.google.com/github/epjf99-rgb/Entregas/blob/main/Javier_TitanicSolucionKaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalar autogluon

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install autogluon

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from autogluon.tabular import TabularDataset, TabularPredictor

# Obtener datos

Subir el archivo titanic.zip que se obtiene de  [Kaggle](https://www.kaggle.com/competitions/titanic/data)

In [ ]:
!unzip -o titanic.zip

unzip:  cannot find or open titanic.zip, titanic.zip.zip or titanic.zip.ZIP.


# Cargar datos

In [ ]:
# usar TabularDataset para cargar al archivo train.csv

train_data = TabularDataset('train.csv')

In [ ]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
label = 'Survived'
train_data[label].describe()

,Survived
count,891.000000
mean,0.383838
std,0.486592
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,1.000000


# Entrenar modelo con AutoGluon

AutoGluon de manera hace Feature Engineering y selecciona el mejor modelo basado en Accuracy

In [ ]:
predictor = TabularPredictor(label=label).fit(train_data)

No path specified. Models will be saved in: "AutogluonModels/ag-20250701_000443"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.11.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sun Mar 30 16:01:29 UTC 2025
CPU Count:          2
Memory Avail:       11.49 GB / 12.67 GB (90.6%)
Disk Space Avail:   64.94 GB / 107.72 GB (60.3%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='experimental' : New in v1.2: Pre-trained foundation model + parallel fits. The absolute best accuracy without consideration for inference speed. Does not support GPU.
	presets='best'         : Maximize accuracy. Recommended for most users. Use in competition

Tabla con resultados del entrenamiento

In [ ]:
leaderboard = predictor.leaderboard(train_data, silent=False)

                  model  score_test  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0        ExtraTreesGini    0.962963   0.815642    accuracy        0.133257       0.099802   0.993915                 0.133257                0.099802           0.993915            1       True          8
1      RandomForestEntr    0.962963   0.815642    accuracy        0.133584       0.102126   1.061491                 0.133584                0.102126           1.061491            1       True          6
2      RandomForestGini    0.962963   0.815642    accuracy        0.148216       0.101073   1.102978                 0.148216                0.101073           1.102978            1       True          5
3        ExtraTreesEntr    0.961841   0.810056    accuracy        0.135701       0.101965   1.017900                 0.135701                0.101965           1.017900            1   

Este es el modelo con mejor calificacion

In [ ]:
best_model = leaderboard.iloc[0]['model']
print(f"Best model: {best_model}")

Best model: ExtraTreesGini


In [ ]:

# Score del mejor modelo en los datos de entrenamiento
scores = predictor.evaluate(train_data, silent=True)
print(f"Scores del mejor modelo en los datos de entrenamiento:\n{scores}")

# Puedes acceder a métricas específicas si lo necesitas
# Por ejemplo, si 'accuracy' es una de las métricas:
if 'accuracy' in scores:
  print(f"Accuracy en los datos de entrenamiento: {scores['accuracy']}")
else:
  print("Accuracy no encontrada en las métricas. Métricas disponibles:", scores.keys())

# También puedes obtener las predicciones del mejor modelo en los datos de entrenamiento
predictions = predictor.predict(train_data)
print(f"\nPrimeras 5 predicciones en los datos de entrenamiento:\n{predictions.head()}")

Scores del mejor modelo en los datos de entrenamiento:
{'accuracy': 0.9326599326599326, 'balanced_accuracy': np.float64(0.9205493241299971), 'mcc': np.float64(0.8574910215178725), 'roc_auc': np.float64(0.9774869779183842), 'f1': 0.908256880733945, 'precision': 0.9519230769230769, 'recall': 0.868421052631579}
Accuracy en los datos de entrenamiento: 0.9326599326599326

Primeras 5 predicciones en los datos de entrenamiento:
0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64


In [ ]:
# Especificar la métrica a optimizar
eval_metric = 'accuracy'

# Define time_limit
time_limit = 60 # seconds, adjust as needed

# Entrenar modelo con AutoGluon, especificando el límite de tiempo y la métrica
# Usamos 'presets' para equilibrar el rendimiento y el tiempo de entrenamiento.
# 'best_quality' tiende a dar mejores resultados pero toma más tiempo.
# 'good_quality' es un buen compromiso.
# predictor = TabularPredictor(label=label, eval_metric=eval_metric).fit(train_data, time_limit=time_limit, presets='best_quality')
predictor = TabularPredictor(label=label, eval_metric=eval_metric).fit(train_data, time_limit=time_limit, presets='good_quality')


# Tabla con resultados del entrenamiento
# Ahora el leaderboard reflejará los modelos entrenados durante el tiempo especificado
leaderboard = predictor.leaderboard(train_data, silent=False)

# Este es el modelo con mejor calificacion (ahora basado en el entrenamiento mejorado)
best_model = leaderboard.iloc[0]['model']
print(f"Best model after enhanced training: {best_model}")

# Score del mejor modelo en los datos de entrenamiento (usando la métrica especificada)
scores = predictor.evaluate(train_data, silent=True)
print(f"Scores del mejor modelo en los datos de entrenamiento:\n{scores}")

# Acceder a la métrica específica 'accuracy'
if eval_metric in scores:
  print(f"{eval_metric} en los datos de entrenamiento: {scores[eval_metric]}")
else:
  print(f"{eval_metric} no encontrada en las métricas. Métricas disponibles:", scores.keys())

# Obtener las predicciones del mejor modelo en los datos de entrenamiento
predictions = predictor.predict(train_data)
print(f"\nPrimeras 5 predicciones en los datos de entrenamiento:\n{predictions.head()}")


No path specified. Models will be saved in: "AutogluonModels/ag-20250701_001302"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.11.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sun Mar 30 16:01:29 UTC 2025
CPU Count:          2
Memory Avail:       11.11 GB / 12.67 GB (87.7%)
Disk Space Avail:   64.88 GB / 107.72 GB (60.2%)
Presets specified: ['good_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid this risk by setting `save_bag_folds=True`.
DyStack is ena

                      model  score_test  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0    LightGBMXT_BAG_L1_FULL    0.867565        NaN    accuracy        0.020557            NaN   0.696677                 0.020557                     NaN           0.696677            1       True          4
1  WeightedEnsemble_L3_FULL    0.867565        NaN    accuracy        0.023417            NaN   0.704866                 0.002861                     NaN           0.008189            3       True          6
2  WeightedEnsemble_L2_FULL    0.867565        NaN    accuracy        0.026046            NaN   0.705224                 0.005489                     NaN           0.008547            2       True          5
3         LightGBMXT_BAG_L1         NaN   0.835017    accuracy             NaN       0.071525  41.135912                      NaN                0.071525          41.13

# Inferir con datos de test.csv

In [ ]:
test_data = TabularDataset('test.csv')

y_pred = predictor.predict(test_data)
y_pred.head(30)

Loaded data from: test.csv | Columns = 11 / 11 | Rows = 418 -> 418


,Survived
0,0
1,1
2,0
3,0
4,1
5,0
6,1
7,0
8,1
9,0


In [ ]:


# # Generar el archivo submission.csv
# Cargar el archivo gender_submission.csv para obtener el formato
submission = pd.read_csv('gender_submission.csv')

# Reemplazar la columna 'Survived' con las predicciones obtenidas
submission['Survived'] = y_pred

# Guardar el archivo de submission
submission.to_csv('submission.csv', index=False)

print("\nArchivo submission.csv generado exitosamente!")
print("Primeras 5 filas del archivo submission.csv:")
print(submission.head())

# Opcional: Verificar el contenido del archivo generado
!head submission.csv


Archivo submission.csv generado exitosamente!
Primeras 5 filas del archivo submission.csv:
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1
PassengerId,Survived
892,0
893,1
894,0
895,0
896,1
897,0
898,1
899,0
900,1


In [ ]:
# prompt: verifica el contenido

# Optional: Verify the content of the generated file
!head submission.csv

PassengerId,Survived
892,0
893,1
894,0
895,0
896,1
897,0
898,1
899,0
900,1


El código realiza las siguientes acciones:

1.  **Importa librerías:** Importa `TabularDataset` y `TabularPredictor` de la biblioteca `autogluon.tabular`, y `pandas` como `pd`.
2.  **Instala AutoGluon:** Se asegura de que `pip` esté actualizado y luego instala la biblioteca `autogluon`.
3.  **Obtiene datos:**
    *   Incluye un comentario que sugiere subir el archivo `titanic.zip` de Kaggle.
    *   Descomprime el archivo `titanic.zip` usando el comando `!unzip -o titanic.zip`.
4.  **Carga datos de entrenamiento:**
    *   Carga el archivo `train.csv` en un objeto `TabularDataset` llamado `train_data`.
    *   Muestra las primeras filas del `train_data`.
    *   Define la columna objetivo (`label`) como `'Survived'`.
    *   Muestra estadísticas descriptivas de la columna `'Survived'`.
5.  **Entrena un modelo con AutoGluon (primera vez):**
    *   Inicializa un `TabularPredictor` con la columna objetivo (`label`).
    *   Entrena el predictor con los datos de entrenamiento (`train_data`). AutoGluon automáticamente realiza ingeniería de características y entrena varios modelos para encontrar el mejor.
    *   Muestra una tabla (`leaderboard`) con los resultados del entrenamiento de los diferentes modelos.
    *   Identifica y imprime el nombre del modelo con la mejor calificación en el `leaderboard`.
    *   Evalúa el mejor modelo en los datos de entrenamiento y muestra las métricas obtenidas.
    *   Intenta imprimir la métrica `'accuracy'` si está disponible.
    *   Obtiene y muestra las primeras 5 predicciones del mejor modelo en los datos de entrenamiento.
6.  **Entrena un modelo con AutoGluon (segunda vez con más control):**
    *   Especifica la métrica a optimizar (`eval_metric`) como `'accuracy'`.
    *   Define un límite de tiempo de 60 segundos (`time_limit`) para el entrenamiento.
    *   Inicializa y entrena un `TabularPredictor` nuevamente, pero esta vez especificando el `eval_metric`, `time_limit` y utilizando el `presets='good_quality'` para equilibrar el rendimiento y el tiempo de entrenamiento.
    *   Vuelve a mostrar el `leaderboard` después de este entrenamiento mejorado.
    *   Identifica e imprime el nombre del mejor modelo después de este entrenamiento.
    *   Evalúa este nuevo mejor modelo en los datos de entrenamiento y muestra las métricas, enfocándose en la métrica especificada (`eval_metric`).
    *   Obtiene y muestra las primeras 5 predicciones de este mejor modelo en los datos de entrenamiento.
7.  **Infiere con datos de test:**
    *   Carga el archivo `test.csv` en un objeto `TabularDataset` llamado `test_data`.
    *   Usa el predictor entrenado para obtener predicciones sobre los datos de prueba (`test_data`).
    *   Muestra las primeras 30 predicciones obtenidas.
8.  **Genera el archivo submission.csv:**
    *   Carga el archivo `gender_submission.csv` (que se asume que tiene el formato requerido para la submission de Kaggle) en un DataFrame de pandas.
    *   Reemplaza la columna `'Survived'` en este DataFrame con las predicciones obtenidas del `test_data` (`y_pred`).
    *   Guarda el DataFrame modificado en un nuevo archivo llamado `submission.csv`, sin incluir el índice.
    *   Imprime mensajes indicando que el archivo `submission.csv` fue generado exitosamente y muestra sus primeras 5 filas.
    *   Utiliza el comando `!head submission.csv` para mostrar el encabezado y las primeras líneas del archivo generado directamente desde el sistema operativo.

En resumen, el código utiliza AutoGluon para entrenar un modelo de clasificación en el dataset del Titanic (para predecir la supervivencia), evalúa el modelo, realiza predicciones sobre un conjunto de datos de prueba y genera un archivo `submission.csv` en el formato esperado por la competición de Kaggle. Se entrena el modelo dos veces, la segunda vez con un límite de tiempo y una métrica de evaluación especificada para un control más fino.

# Exportar para competir en Kaggle

In [ ]:
# prompt: generar archivo submission.csv , primera columna PassengerId , segunda columna Survived

import pandas as pd

# Create a DataFrame with PassengerId and Survived columns
submission_df = pd.DataFrame({'PassengerId': test_data['PassengerId'], 'Survived': y_pred})

# Save the DataFrame to a CSV file
submission_df.to_csv('submissionEJPF.csv', index=False)

print("submission.csv created successfully!")

submission.csv created successfully!
